In [ ]:
!pip uninstall -y qiskit-aer
!pip install qiskit-aer-gpu --upgrade

In [11]:
from qiskit_aer import AerSimulator

backend = AerSimulator(
    method='statevector',
    device='GPU'
)

print(backend.available_devices())

['GPU']


In [ ]:
!nvidia-smi

In [ ]:
!pip install cuquantum

---

# Experiments A1.1 & A1.2 — Baseline Characterisation

**GPU strategy:** A custom vectorised density-matrix runner (`gpu_batch_bb84`) processes
all *T* trials simultaneously as a `(T, N, 2, 2)` tensor using **CuPy** on the Tesla T4 GPU
(or NumPy as a transparent CPU fallback). This replaces the per-qubit Qiskit loops and
gives throughput in the tens-of-millions-of-qubits-per-second range.

**Run order:**
1. Environment & CuPy setup
2. bb84 package path
3. `gpu_batch_bb84` runner definition
4. A1.1 — Ideal BB84 experiment + visualisation
5. A1.2 — Channel length sweep experiment + visualisation + summary table

In [13]:
# ── Install CuPy for GPU-accelerated NumPy operations ────────────────────────
import subprocess, sys

def _try_cupy():
    try:
        import cupy as cp
        _ = cp.array([1.0])    # trigger CUDA context
        return cp
    except Exception:
        return None

cp_mod = _try_cupy()
if cp_mod is None:
    print("CuPy not found — installing cupy-cuda12x …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    cp_mod = _try_cupy()

GPU = cp_mod is not None

import numpy as np
import math, time, warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")

if GPU:
    import cupy as cp
    xp    = cp
    _dev  = cp.cuda.runtime.getDeviceProperties(0)
    _mem  = cp.cuda.runtime.memGetInfo()[1] / 1e9
    print(f"✅  GPU  : {_dev['name'].decode()}")
    print(f"    VRAM : {_mem:.1f} GB total")
else:
    xp = np
    print("⚠  CuPy unavailable — using NumPy (CPU fallback)")

plt.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})
sns.set_palette("husl")
print("Imports OK.")

✅  GPU  : Tesla T4
    VRAM : 15.6 GB total
Imports OK.


In [ ]:
# ── bb84 package path ────────────────────────────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
    # ⚠️  Update this to where QKD_BB84/ lives on your Drive
    BB84_ROOT = "/content/drive/MyDrive/QKD_BB84"
else:
    BB84_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if BB84_ROOT not in sys.path:
    sys.path.insert(0, BB84_ROOT)

from bb84.config import SimulationConfig
from bb84.fast   import fast_run_simulation
from bb84.core   import estimate_qber

print(f"BB84 root : {BB84_ROOT}")
print("bb84 package imported.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GPU-ACCELERATED BATCH BB84 RUNNER
# Processes T trials × N qubits simultaneously as (T, N, 2, 2) tensors.
# Uses CuPy on GPU or NumPy on CPU transparently via the global `xp`.
# ════════════════════════════════════════════════════════════════════════════

_sqrt2inv = 1.0 / math.sqrt(2)

def _gates(xp_):
    X = xp_.array([[0, 1], [1, 0]], dtype=xp_.complex128)
    H = xp_.array([[1, 1], [1, -1]], dtype=xp_.complex128) * _sqrt2inv
    return X, H

def _apply_masked(rhos, U, mask, xp_):
    """Apply gate UρU† for all entries where mask (T, N) is True."""
    transformed = xp_.einsum("ij,tnjk,mk->tnim", U, rhos, U.conj())
    return xp_.where(mask[:, :, None, None], transformed, rhos)

def _measure_z(rhos, rng_, xp_):
    """Sample Z-basis outcome. P(|1⟩) = ρ[1,1].real. Returns (T, N) int."""
    p1 = rhos[:, :, 1, 1].real.clip(0.0, 1.0)
    return (rng_.random(p1.shape) < p1).astype(xp_.int32)

def gpu_batch_bb84(
    n_qubits:    int,
    n_trials:    int,
    seed:        int   = 42,
    fiber_km:    float = 0.0,
    eta_det:     float = 1.0,
    alpha_db_km: float = 0.2,
) -> dict:
    """
    Vectorized BB84 — all n_trials run in parallel on GPU (or CPU fallback).

    Parameters
    ----------
    n_qubits    : raw qubits Alice transmits per trial
    n_trials    : independent runs to batch
    fiber_km    : fiber length in km; 0 = ideal (no loss)
    eta_det     : detector efficiency in [0, 1]
    alpha_db_km : fiber attenuation dB/km (SMF-28 default = 0.2)

    Returns dict with (n_trials,) numpy arrays:
        qber, sift_rate, n_received, n_sifted, n_key, runtime_s
    """
    t0 = time.perf_counter()
    T, N = n_trials, n_qubits

    X, H = _gates(xp)
    rng_a = xp.random.default_rng(seed)
    rng_b = xp.random.default_rng(seed + 1)
    rng_c = xp.random.default_rng(seed + 2)

    alice_bits  = rng_a.integers(0, 2, (T, N))
    alice_bases = rng_a.integers(0, 2, (T, N))
    bob_bases   = rng_b.integers(0, 2, (T, N))

    # Init |0⟩⟨0| for every (trial, qubit)
    rhos = xp.zeros((T, N, 2, 2), dtype=xp.complex128)
    rhos[:, :, 0, 0] = 1.0

    # Alice encodes: X if bit=1, H if basis=X
    rhos = _apply_masked(rhos, X, alice_bits  == 1, xp)
    rhos = _apply_masked(rhos, H, alice_bases == 1, xp)

    # Fiber loss + detector efficiency
    if fiber_km > 0 or eta_det < 1.0:
        eta_total = (10 ** (-alpha_db_km * fiber_km / 10)) * eta_det
        detected  = rng_c.random((T, N)) < eta_total
    else:
        detected = xp.ones((T, N), dtype=bool)

    # Bob measures: H if basis=X, then Z-measure
    rhos     = _apply_masked(rhos, H, bob_bases == 1, xp)
    bob_bits = _measure_z(rhos, rng_b, xp)

    # Sifting: same basis AND photon detected
    same_basis = (alice_bases == bob_bases) & detected
    n_received = detected.sum(axis=1).astype(xp.float64)
    n_sifted   = same_basis.sum(axis=1).astype(xp.float64)
    errors     = (same_basis & (alice_bits != bob_bits)).sum(axis=1).astype(xp.float64)

    qber      = errors / xp.maximum(n_sifted, 1)
    sift_rate = n_sifted / N
    n_key     = xp.floor(n_sifted * 0.55).astype(xp.int64)   # 45% for QBER check

    to_np = lambda a: a.get() if GPU else np.asarray(a)
    return dict(
        qber       = to_np(qber),
        sift_rate  = to_np(sift_rate),
        n_received = to_np(n_received),
        n_sifted   = to_np(n_sifted),
        n_key      = to_np(n_key),
        runtime_s  = time.perf_counter() - t0,
    )

# ── Smoke test ────────────────────────────────────────────────────────────────
_t = gpu_batch_bb84(n_qubits=1_000, n_trials=10, seed=0)
print(f"Smoke test ({'GPU' if GPU else 'CPU'}):  "
      f"QBER={_t['qber'].mean():.4f}  sift={_t['sift_rate'].mean():.4f}")
print("Batch runner ready.")

---

## Experiment A1.1 — Ideal BB84 Baseline (Zero Noise)

**Objective:** Establish a clean reference. QBER must be statistically indistinguishable from 0; sifting ratio ≈ 0.5.

| Parameter | Value |
|-----------|-------|
| N (raw qubits) | 1 000 · 10 000 · 100 000 |
| Noise model | None (ideal channel) |
| Eve present | No |
| Trials per N | 100 |
| QBER sample fraction | 45% of sifted key |

**Expected outcomes:**  
- QBER = 0.000 for every trial (deterministic matching bases → no errors)  
- Sifting ratio = 0.500 ± √(0.25/N)  
- 95% CI upper bound shrinks as **1/N** (Wilson formula, 0 errors)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# A1.1 — Run ideal BB84 for N ∈ {1k, 10k, 100k}
# ════════════════════════════════════════════════════════════════════════════

N_VALUES  = [1_000, 10_000, 100_000]
N_TRIALS  = 100
SEED_BASE = 42

a11 = {}

for N in N_VALUES:
    print(f"\n{'─'*58}")
    print(f"  N = {N:>7,}  |  {N_TRIALS} trials  |  {'GPU' if GPU else 'CPU'}")

    res = gpu_batch_bb84(n_qubits=N, n_trials=N_TRIALS, seed=SEED_BASE)

    qber      = res["qber"]
    sift_rate = res["sift_rate"]
    n_key     = res["n_key"]
    rt        = res["runtime_s"]

    # Wilson CI upper bound for 0 errors in ~n_sample bits
    n_sample  = max(1, int(N * 0.5 * 0.45))
    z         = 1.96
    ci_up     = (z**2 / (2 * n_sample)) / (1 + z**2 / n_sample)

    a11[N] = dict(
        qber_all  = qber,
        sift_all  = sift_rate,
        nkey_all  = n_key,
        mean_qber = qber.mean(),
        std_qber  = qber.std(),
        mean_sift = sift_rate.mean(),
        std_sift  = sift_rate.std(),
        mean_key  = n_key.mean(),
        ci_up     = ci_up,
        ci_up_pct = ci_up * 100,
        runtime_s = rt,
    )

    print(f"  QBER       : {qber.mean():.2e} ± {qber.std():.2e}"
          f"   95% CI upper: {ci_up*100:.3f}%")
    print(f"  Sift ratio : {sift_rate.mean():.4f} ± {sift_rate.std():.4f}"
          f"   (expected 0.5)")
    print(f"  Key bits   : {n_key.mean():.0f} bits/run")
    print(f"  Runtime    : {rt:.3f}s  →  {N_TRIALS/rt:.1f} trials/s"
          f"  |  {N_TRIALS*N/rt/1e6:.2f}M qubits/s")

print("\n✅  A1.1 complete.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# A1.1 — Visualisation
# ════════════════════════════════════════════════════════════════════════════

COLORS = ["#2196F3", "#4CAF50", "#FF5722"]
LABELS = ["N = 1k", "N = 10k", "N = 100k"]

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.40)

# ── 1. QBER distribution (violin + strip) ────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
for i, N in enumerate(N_VALUES):
    data  = a11[N]["qber_all"] * 100
    parts = ax1.violinplot(data, positions=[i + 1], showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(COLORS[i]); pc.set_alpha(0.55)
    for k in ("cbars", "cmins", "cmaxes", "cmedians"):
        if k in parts:
            parts[k].set_color(COLORS[i])
    jitter = np.random.default_rng(i).uniform(-0.07, 0.07, len(data))
    ax1.scatter(np.full(len(data), i + 1) + jitter, data,
                c=COLORS[i], alpha=0.45, s=20, zorder=3)

ax1.axhline(0,  color="k",      lw=1.8, ls="--", label="Ideal QBER = 0%")
ax1.axhline(5,  color="orange", lw=1,   ls=":",  label="Warning threshold 5%")
ax1.axhline(11, color="red",    lw=1,   ls=":",  label="Abort threshold 11%")
ax1.set_xticks([1, 2, 3]); ax1.set_xticklabels(LABELS, fontsize=11)
ax1.set_ylabel("QBER (%)", fontsize=11)
ax1.set_ylim(-0.5, 14)
ax1.set_title("QBER Distribution — 100 Independent Trials per N", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9, loc="upper right")

# ── 2. Sifting ratio histogram ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
for i, N in enumerate(N_VALUES):
    ax2.hist(a11[N]["sift_all"], bins=15, alpha=0.55,
             color=COLORS[i], label=LABELS[i], edgecolor="white", linewidth=0.5)
ax2.axvline(0.5, color="k", lw=1.8, ls="--", label="Expected 0.5")
ax2.set_xlabel("Sifting Ratio", fontsize=10); ax2.set_ylabel("Count", fontsize=10)
ax2.set_title("Sifting Ratio\nDistribution", fontsize=11, fontweight="bold")
ax2.legend(fontsize=8)

# ── 3. 95% CI upper bound vs N (log-log) ─────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ns        = np.array(N_VALUES, dtype=float)
ci_uppers = np.array([a11[N]["ci_up_pct"] for N in N_VALUES])
ax3.loglog(ns, ci_uppers, "o-", color="#9C27B0", lw=2, ms=9, zorder=5)
for N, ci in zip(N_VALUES, ci_uppers):
    ax3.annotate(f"{ci:.3f}%", (N, ci), textcoords="offset points",
                 xytext=(6, 4), fontsize=8.5, color="#9C27B0")
n_ref = np.logspace(2.9, 5.1, 100)
ax3.loglog(n_ref, ci_uppers[0] * np.sqrt(N_VALUES[0] / n_ref),
           "k--", lw=1, alpha=0.45, label="∝ N⁻⁰·⁵")
ax3.set_xlabel("N (raw qubits)", fontsize=10)
ax3.set_ylabel("95% CI Upper Bound (%)", fontsize=10)
ax3.set_title("QBER Uncertainty\n(Wilson CI, 0 errors)", fontsize=11, fontweight="bold")
ax3.legend(fontsize=9); ax3.grid(True, which="both", alpha=0.3)

# ── 4. Key throughput bar chart ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
mean_keys = [a11[N]["mean_key"] for N in N_VALUES]
bars = ax4.bar(LABELS, mean_keys, color=COLORS, edgecolor="white", linewidth=0.8)
ax4.set_yscale("log")
for bar, val in zip(bars, mean_keys):
    ax4.text(bar.get_x() + bar.get_width() / 2, val * 1.15,
             f"{val:.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax4.set_ylabel("Key Bits / Run", fontsize=10)
ax4.set_title("Sifted-Key Throughput\n(bits per run)", fontsize=11, fontweight="bold")

# ── 5. Summary table ──────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis("off")
rows = [
    [f"{N:,}", f"{a11[N]['mean_qber']:.2e}",
     f"{a11[N]['mean_sift']:.4f}",
     f"{a11[N]['mean_key']:.0f}",
     f"{a11[N]['ci_up_pct']:.3f}%",
     f"{a11[N]['runtime_s']:.2f}s"]
    for N in N_VALUES
]
tbl = ax5.table(
    cellText=rows,
    colLabels=["N", "QBER", "Sift Ratio", "Key Bits", "CI Upper", "Time"],
    cellLoc="center", loc="center", bbox=[0, 0.15, 1, 0.75],
)
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1565C0"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2:
        cell.set_facecolor("#E3F2FD")
ax5.set_title("Summary Statistics", fontsize=11, fontweight="bold", pad=16)

fig.suptitle("A1.1 — Ideal BB84 Baseline Characterisation",
             fontsize=14, fontweight="bold", y=1.01)
plt.savefig("a1_1_baseline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: a1_1_baseline.png")

---

## Experiment A1.2 — Channel Length Sweep

**Objective:** Relate fiber distance L (km) to detection rate and key rate.

| Parameter | Value |
|-----------|-------|
| Distances L (km) | 0, 10, 25, 50, 75, 100, 150, 200 |
| Attenuation α | 0.2 dB/km (SMF-28 at 1550 nm) |
| Detector efficiency η\_det | 0.9 (InGaAs APD) |
| N per trial | 10 000 raw qubits |
| Trials per distance | 50 |

**Metrics:** sifted-key rate R\_sift, secure-key rate R\_sec, transmittance η = 10^(−αL/10)

**Key insight:** Fiber loss reduces key *rate* but does **not** introduce QBER — this distinguishes passive loss from active eavesdropping.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# A1.2 — Channel Length Sweep
# ════════════════════════════════════════════════════════════════════════════

DISTANCES  = [0, 10, 25, 50, 75, 100, 150, 200]   # km
N_RAW      = 10_000
N_TRIALS_D = 50
ETA_DET    = 0.9      # InGaAs APD detector efficiency
ALPHA      = 0.2      # dB/km  (SMF-28 at 1550 nm)

def hbin(e: float) -> float:
    """Binary entropy  H(e) = −e log₂e − (1−e) log₂(1−e)."""
    if e <= 0.0 or e >= 1.0:
        return 0.0
    return -e * math.log2(e) - (1.0 - e) * math.log2(1.0 - e)

a12 = {}

print(f"{'L (km)':>8}  {'η_fiber':>8}  {'η_total':>8}  "
      f"{'R_sift (sim)':>13}  {'R_sec (sim)':>12}  {'n_recv':>8}  {'QBER':>8}")
print("─" * 82)

for L in DISTANCES:
    eta_f = 10 ** (-ALPHA * L / 10)
    eta_t = eta_f * ETA_DET

    res = gpu_batch_bb84(
        n_qubits    = N_RAW,
        n_trials    = N_TRIALS_D,
        seed        = SEED_BASE + L * 7,
        fiber_km    = L,
        eta_det     = ETA_DET,
        alpha_db_km = ALPHA,
    )

    qber_arr  = res["qber"]
    sift_arr  = res["sift_rate"]
    nrecv_arr = res["n_received"]
    nsift_arr = res["n_sifted"]
    nkey_arr  = res["n_key"]

    mean_qber = qber_arr.mean()
    mean_sift = sift_arr.mean()
    r_sec_sim = max(0.0, mean_sift * (1.0 - 2.0 * hbin(mean_qber)))

    a12[L] = dict(
        eta_fiber  = eta_f,
        eta_total  = eta_t,
        qber_all   = qber_arr,
        sift_all   = sift_arr,
        nrecv_all  = nrecv_arr,
        nsift_all  = nsift_arr,
        nkey_all   = nkey_arr,
        mean_qber  = mean_qber,
        mean_sift  = mean_sift,
        std_sift   = sift_arr.std(),
        r_sift_th  = eta_t / 2,
        r_sec_sim  = r_sec_sim,
        mean_nrecv = nrecv_arr.mean(),
        mean_nsift = nsift_arr.mean(),
        mean_nkey  = nkey_arr.mean(),
        runtime_s  = res["runtime_s"],
    )

    print(f"{L:>8}  {eta_f:>8.4f}  {eta_t:>8.4f}  "
          f"{mean_sift:>13.4f}  {r_sec_sim:>12.4f}  "
          f"{nrecv_arr.mean():>8.1f}  {mean_qber:>8.4f}")

print("\n✅  A1.2 complete.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# A1.2 — Visualisation
# ════════════════════════════════════════════════════════════════════════════

Ls       = np.array(DISTANCES, dtype=float)
Ls_fine  = np.linspace(0, 205, 500)

eta_th_fine   = ETA_DET * 10 ** (-ALPHA * Ls_fine / 10)
rsift_th_fine = eta_th_fine / 2
rsec_th_fine  = rsift_th_fine   # QBER=0 → 1 − 2H(0) = 1

sim_eta      = np.array([a12[L]["mean_nrecv"] / N_RAW for L in DISTANCES])
sim_rsift    = np.array([a12[L]["mean_sift"]           for L in DISTANCES])
sim_rsec     = np.array([a12[L]["r_sec_sim"]            for L in DISTANCES])
sim_rsift_sd = np.array([a12[L]["std_sift"]             for L in DISTANCES])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"A1.2 — Fiber Channel Length Sweep\n"
    f"(N={N_RAW:,} / trial, {N_TRIALS_D} trials / distance, "
    f"η_det={ETA_DET}, α={ALPHA} dB/km)",
    fontsize=13, fontweight="bold",
)

# ── 1. Transmittance vs Distance ─────────────────────────────────────────────
ax = axes[0, 0]
ax.semilogy(Ls_fine, eta_th_fine, "b-", lw=2, label="Theory  η = η_det · 10^(−αL/10)")
ax.semilogy(Ls, sim_eta, "ro", ms=7, zorder=5, label="Simulation (mean)")
for L_m, sty in [(50, "--"), (100, "--"), (150, ":")]:
    eta_m = ETA_DET * 10 ** (-ALPHA * L_m / 10)
    ax.axvline(L_m, color="gray", lw=1, ls=sty, alpha=0.6)
    ax.text(L_m + 2, eta_m * 2.5, f"{L_m} km\nη={eta_m:.3f}",
            fontsize=7.5, color="gray", va="bottom")
ax.set_xlabel("L (km)", fontsize=11); ax.set_ylabel("Transmittance η", fontsize=11)
ax.set_title("Channel Transmittance vs Distance", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3); ax.set_xlim(-2, 210)

# ── 2. Key Rate vs Distance ──────────────────────────────────────────────────
ax = axes[0, 1]
ax.semilogy(Ls_fine, rsift_th_fine, "b-",  lw=2, label="R_sift  (theory)")
ax.semilogy(Ls_fine, rsec_th_fine,  "g--", lw=2, label="R_sec   (theory, QBER=0)")
ax.semilogy(Ls, sim_rsift, "rs", ms=8, zorder=5, label="R_sift  (simulation)")
ax.fill_between(
    Ls,
    np.maximum(sim_rsift - sim_rsift_sd, 1e-6),
    sim_rsift + sim_rsift_sd,
    color="red", alpha=0.15, label="±1σ",
)
ax.set_xlabel("L (km)", fontsize=11)
ax.set_ylabel("Key Rate (bits / raw qubit)", fontsize=11)
ax.set_title("Key Rate vs Channel Length", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3); ax.set_xlim(-2, 210)

# ── 3. Received Photons per Trial ────────────────────────────────────────────
ax = axes[1, 0]
nrecv_th = N_RAW * ETA_DET * 10 ** (-ALPHA * Ls_fine / 10)
ax.semilogy(Ls_fine, nrecv_th, "b-", lw=2, label="Theory")
ax.semilogy(Ls, np.array([a12[L]["mean_nrecv"] for L in DISTANCES]),
            "ro", ms=7, zorder=5, label="Simulation")
ax.axhline(1, color="red", lw=1.2, ls="--", label="1 photon / trial")
km_1ph = -10 / ALPHA * math.log10(1 / (N_RAW * ETA_DET))
if 0 < km_1ph < 210:
    ax.axvline(km_1ph, color="red", lw=1.5, ls=":", alpha=0.7)
    ax.text(km_1ph + 3, 1.5, f"<1 photon\nat {km_1ph:.0f} km",
            color="red", fontsize=8.5, va="bottom")
ax.set_xlabel("L (km)", fontsize=11)
ax.set_ylabel("Received photons / trial", fontsize=11)
ax.set_title(f"Photon Arrival Rate  (N={N_RAW:,} raw)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3); ax.set_xlim(-2, 210)

# ── 4. QBER vs Distance ──────────────────────────────────────────────────────
ax = axes[1, 1]
qber_means = np.array([a12[L]["mean_qber"] * 100 for L in DISTANCES])
qber_stds  = np.array([a12[L]["qber_all"].std() * 100 for L in DISTANCES])
ax.errorbar(Ls, qber_means, yerr=qber_stds,
            fmt="bo-", ms=7, capsize=5, lw=1.8, label="QBER (mean ± σ)")
ax.axhline(0,  color="k",      lw=1.5, ls="--", label="Ideal = 0%")
ax.axhline(5,  color="orange", lw=1,   ls=":",  label="Warning 5%")
ax.axhline(11, color="red",    lw=1,   ls=":",  label="Abort 11%")
ax.text(80, 7.0,
        "Fiber loss ≠ eavesdropping\nLoss reduces rate, not QBER",
        fontsize=9, style="italic", color="navy",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.85))
ax.set_xlabel("L (km)", fontsize=11); ax.set_ylabel("QBER (%)", fontsize=11)
ax.set_title("QBER vs Distance\n(key insight: loss ≠ errors)", fontsize=11, fontweight="bold")
ax.set_ylim(-1, 15); ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim(-5, 210)

plt.tight_layout()
plt.savefig("a1_2_channel_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: a1_2_channel_sweep.png")

In [ ]:
# ── A1.2 Summary Table ───────────────────────────────────────────────────────
print("═" * 90)
print(f"{'A1.2  Summary Table':^90}")
print("═" * 90)
print(f"{'L (km)':>7}  {'η_fiber':>8}  {'η_total':>8}  "
      f"{'n_recv':>8}  {'n_sift':>8}  {'n_key':>7}  "
      f"{'R_sift':>8}  {'R_sec':>8}  {'QBER (%)':>9}")
print("─" * 90)
for L in DISTANCES:
    r = a12[L]
    print(
        f"{L:>7}  {r['eta_fiber']:>8.4f}  {r['eta_total']:>8.4f}  "
        f"{r['mean_nrecv']:>8.1f}  {r['mean_nsift']:>8.1f}  "
        f"{r['mean_nkey']:>7.1f}  "
        f"{r['mean_sift']:>8.4f}  {r['r_sec_sim']:>8.4f}  "
        f"{r['mean_qber']*100:>9.4f}"
    )
print("═" * 90)
print(f"\nN = {N_RAW:,} raw qubits/trial  |  η_det = {ETA_DET}  |  α = {ALPHA} dB/km")
print("R_sift = n_sifted / N_raw")
print("R_sec  = R_sift × (1 − 2H(QBER))  [Mayers / Lo-Chau asymptotic bound]")